# Train Model with DPO

### imports

In [ ]:
# TODO

### Login huggingface and load data

In [ ]:
# TODO

In [ ]:
dataset = # TODO

### load model

We are gonna use the unsloth library which is faster at fine-tuning job. Before that we need to connect with T4 GPU from the runtime menu.

In [ ]:
# TODO

Install the required library to use the unsloth

In [ ]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# We have to check which Torch version for Xformers (2.3 -> 0.0.27)
from torch import __version__; from packaging.version import Version as V
# Correctly determine and install xformers based on the installed torch version
xformers_version = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers>=0.0.30"
!pip install --no-deps {xformers_version} trl peft accelerate bitsandbytes triton

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from datasets import load_dataset, Dataset
import pandas as pd
import json, yaml
import torch

In [ ]:
!pip install -U bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

# Set model name
model_name = # TODO  # Use this exact quantized model

# Load model and tokenizer with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,       # You can increase this if needed
    load_in_4bit = True,         # Efficient memory usage
    load_in_8bit = False,        # Keep False if using 4bit
    full_finetuning = False,     # Set True only if you're planning full finetuning
    # token = "hf_...",          # Only needed for gated models
)

# Ensure pad_token is set properly
tokenizer.pad_token = tokenizer.eos_token


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = # TODO
    target_modules = # TODO
    lora_alpha = # TODO
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

### generate title with base model

In [ ]:
def format_chat_prompt(user_input, system_message="You are a helpful assistant."):
    """
    Formats user input into the chat template format with <|im_start|> and <|im_end|> tags.

    Args:
        user_input (str): The input text from the user.

    Returns:
        str: Formatted prompt for the model.
    """

    # Format user message
    user_prompt = # TODO

    # Start assistant's turn
    assistant_prompt = # TODO

    # Combine prompts
    formatted_prompt = # TODO

    return formatted_prompt

In [ ]:
from transformers import pipeline

# Set up text generation pipeline (DO NOT set device manually)
generator = # TODO

# Example prompt — update this based on your actual structure
prompt = # TODO

# Generate output
outputs = # TODO

# Print the result
print(outputs[0]['generated_text'])

### train model

In [ ]:
from trl import DPOTrainer, DPOConfig

# Define DPO training arguments
training_args = DPOConfig(
    # TODO
    seed=42,
)

# Initialize DPOTrainer
trainer = DPOTrainer(
    # TODO
)

# Start training
trainer.train()


### use fine-tuned model

In [ ]:
# Load the fine-tuned model
ft_model = # TODO

In [ ]:
# Set up text generation pipeline (DO NOT set device manually)
generator = # TODO

# Example prompt — update this based on your actual structure
prompt = # TODO

# Generate output

# Print the result
print(outputs[0]['generated_text'])